# LRTIA v2 - Fixed Distance Sampling

**Key fix:** Only use target positions that can test ALL distances.
This ensures we're comparing the same text regions across all distance conditions.

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from typing import List, Dict
import random
import re
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if USE_4BIT and device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )
model.eval()
print(f"Loaded! {model.num_parameters():,} parameters")

In [ ]:
# Longer passages (~600-700 tokens each)
PASSAGES = [
    """The Amazon rainforest is the world's largest tropical rainforest, covering approximately 5.5 million square kilometers. It spans across nine countries in South America, with the majority located in Brazil. The forest is often called the lungs of the Earth because it produces about 20 percent of the world's oxygen. Scientists estimate that the Amazon is home to approximately 390 billion individual trees, representing around 16,000 different species. The biodiversity found here is unmatched anywhere else on the planet. The Amazon River, which flows through the heart of the forest, is the second longest river in the world. It carries more water than any other river system, accounting for roughly 20 percent of all freshwater that flows into the world's oceans. The river and its tributaries support an incredible variety of aquatic life, including pink river dolphins, giant otters, and piranhas. More than 3,000 species of fish have been identified in its waters, with scientists believing many more remain undiscovered. Indigenous peoples have inhabited the Amazon for at least 11,000 years. Today, approximately 400 distinct indigenous groups live within the forest, speaking more than 300 different languages. These communities have developed sophisticated knowledge of the forest's medicinal plants and sustainable harvesting techniques. Their traditional practices have helped maintain the forest's ecological balance for generations. Deforestation poses the greatest threat to the Amazon's survival. Each year, thousands of square kilometers of forest are cleared for cattle ranching, soybean farming, and logging. This destruction releases massive amounts of carbon dioxide into the atmosphere, contributing to global climate change. Scientists warn that continued deforestation could push the Amazon past a tipping point, transforming large portions of the rainforest into savanna. Conservation efforts have intensified in recent decades. Brazil has established numerous protected areas and indigenous territories that limit development. International organizations have invested billions of dollars in preservation programs.""",
    
    """Marie Curie was born Maria Sklodowska in Warsaw, Poland, on November 7, 1867. She grew up in a family that valued education, despite living under Russian occupation that restricted Polish culture and language. Her father was a physics and mathematics teacher, and her mother ran a prestigious boarding school. From an early age, Marie showed exceptional intelligence and a passion for learning. At the time, women in Poland were not permitted to attend university. Marie worked as a governess for several years to help fund her older sister's medical studies in Paris. In exchange, her sister later helped support Marie's own education. In 1891, at age 24, Marie finally moved to Paris to study physics and mathematics at the Sorbonne, one of the few European universities that admitted women. In Paris, Marie lived in a tiny attic apartment and often survived on little more than bread and chocolate. Despite these hardships, she excelled in her studies, earning degrees in both physics and mathematics. It was during this time that she met Pierre Curie, a professor at the School of Physics. They shared a passion for scientific research and were married in 1895. Their partnership would prove to be one of the most productive collaborations in scientific history. Marie became fascinated by Henri Becquerel's recent discovery of mysterious rays emitted by uranium. She decided to investigate these rays for her doctoral thesis. Working in a converted shed with minimal equipment, she and Pierre made breakthrough after breakthrough. Marie coined the term radioactivity to describe the phenomenon. In 1898, she announced the discovery of two new elements: polonium, named after her homeland, and radium. In 1903, Marie Curie became the first woman to win a Nobel Prize, sharing the physics prize with Pierre and Henri Becquerel. Tragically, Pierre was killed in a street accident in 1906. Despite her grief, Marie continued their research and took over his teaching position at the Sorbonne. She became the university's first female professor. In 1911, she won a second Nobel Prize, this time in chemistry.""",

    """Albert Einstein was born in Ulm, Germany, on March 14, 1879. His family moved to Munich when he was an infant, where his father and uncle founded an electrical equipment company. Young Albert showed an early fascination with invisible forces, reportedly becoming captivated by a compass at age five. He wondered what unseen power could make the needle always point north. Einstein struggled with the rigid, authoritarian style of German schools. He excelled in physics and mathematics but chafed at rote memorization and strict discipline. At age 15, he dropped out of school and moved to Switzerland, where he eventually gained admission to the Swiss Federal Polytechnic in Zurich. After graduating, Einstein could not find a teaching position and took a job at the Swiss Patent Office in Bern. This seemingly mundane work proved fortuitous. The job left him time to think about physics, and he later credited the patent office with teaching him to question assumptions and express ideas precisely. In 1905, Einstein published four groundbreaking papers that would revolutionize physics. One explained the photoelectric effect using quantum theory, work that would later earn him the Nobel Prize. Another provided evidence for the existence of atoms by explaining Brownian motion. The third introduced special relativity, fundamentally changing our understanding of space and time. The fourth derived the famous equation E equals mc squared, revealing the equivalence of mass and energy. Einstein spent the next decade developing his general theory of relativity, which reimagined gravity not as a force but as a curvature of spacetime caused by mass and energy. The theory made startling predictions: that light would bend around massive objects, that time would slow in strong gravitational fields, and that the universe itself might be expanding. When a 1919 solar eclipse confirmed his prediction of light bending, Einstein became an international celebrity overnight.""",
]

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

def create_corpus(passages, n_shuffles=4):
    docs = []
    for i, passage in enumerate(passages):
        docs.append({"doc_id": f"intact_{i}", "population": "intact", "text": " ".join(passage.split())})
        sentences = split_sentences(passage)
        for s in range(n_shuffles):
            rng = random.Random(42 + i * 100 + s)
            shuffled = sentences.copy()
            rng.shuffle(shuffled)
            docs.append({"doc_id": f"shuffled_{i}_{s}", "population": "shuffled", "text": " ".join(shuffled)})
    return docs

corpus = create_corpus(PASSAGES, n_shuffles=5)
print(f"Created {len(corpus)} documents")
lengths = [len(tokenizer.encode(d['text'])) for d in corpus]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")

## KEY FIX: Controlled Distance Sampling

Only test targets that can accommodate ALL distances, so we compare identical text regions.

In [ ]:
@torch.no_grad()
def get_token_logprobs(token_ids: List[int], positions: List[int]) -> np.ndarray:
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    log_probs = torch.log_softmax(outputs.logits[0], dim=-1)
    results = []
    for pos in positions:
        if pos + 1 < len(token_ids):
            results.append(log_probs[pos, token_ids[pos + 1]].cpu().item())
    return np.array(results)


def mask_span(token_ids: List[int], start: int, end: int, mask_id: int) -> List[int]:
    result = token_ids.copy()
    for i in range(start, min(end, len(result))):
        result[i] = mask_id
    return result


def analyze_document_controlled(
    text: str,
    distances: List[int],
    span_width: int = 16,
    region_length: int = 20,
    n_targets: int = 5,
) -> List[Dict]:
    """
    CONTROLLED analysis: only use targets that can test ALL distances.
    This ensures fair comparison across distance conditions.
    """
    token_ids = tokenizer.encode(text)
    n_tokens = len(token_ids)
    mask_id = tokenizer.eos_token_id or 0
    
    max_distance = max(distances)
    
    # Target must have enough room for max distance BEFORE it
    # and enough room for region_length AFTER it
    min_target_pos = max_distance + span_width + 10  # buffer
    max_target_pos = n_tokens - region_length - 1
    
    if min_target_pos >= max_target_pos:
        return []  # Document too short
    
    # Select evenly spaced targets in valid range
    valid_range = max_target_pos - min_target_pos
    if valid_range < n_targets:
        target_starts = list(range(min_target_pos, max_target_pos))
    else:
        step = valid_range // n_targets
        target_starts = [min_target_pos + i * step for i in range(n_targets)]
    
    results = []
    
    for target_start in target_starts:
        target_end = target_start + region_length
        target_positions = list(range(target_start, min(target_end, n_tokens - 1)))
        
        if len(target_positions) < 5:
            continue
        
        # Original predictions
        orig_lp = get_token_logprobs(token_ids, target_positions)
        orig_nll = -np.mean(orig_lp)
        
        for distance in distances:
            span_center = target_start - distance
            span_start = span_center - span_width // 2
            span_end = span_start + span_width
            
            # Strict check: span must be fully valid
            if span_start < 0 or span_end >= target_start - 5:
                continue
            
            masked_ids = mask_span(token_ids, span_start, span_end, mask_id)
            masked_lp = get_token_logprobs(masked_ids, target_positions)
            masked_nll = -np.mean(masked_lp)
            
            delta_nll = masked_nll - orig_nll
            
            results.append({
                "distance": distance,
                "delta_nll": delta_nll,
                "target_pos": target_start,
                "span_start": span_start,
            })
    
    return results

In [ ]:
# Use shorter max distance to get more valid samples
DISTANCES = [32, 64, 96, 128, 160, 192, 224, 256]
SPAN_WIDTH = 16
REGION_LENGTH = 20
N_TARGETS = 8  # targets per document

all_results = []

for doc in tqdm(corpus, desc="Analyzing"):
    doc_results = analyze_document_controlled(
        doc["text"],
        distances=DISTANCES,
        span_width=SPAN_WIDTH,
        region_length=REGION_LENGTH,
        n_targets=N_TARGETS,
    )
    for r in doc_results:
        r["doc_id"] = doc["doc_id"]
        r["population"] = doc["population"]
        all_results.append(r)

df = pd.DataFrame(all_results)
print(f"\nCollected {len(df)} measurements")
print("\nSamples per condition:")
print(df.groupby(['population', 'distance']).size().unstack(fill_value=0))

In [ ]:
print("=" * 70)
print(f"RESULTS (Controlled Sampling): {MODEL_NAME}")
print("=" * 70)

pivot = df.pivot_table(values='delta_nll', index='distance', columns='population', aggfunc='mean')
print("\nMean delta_nll by distance:")
print(pivot.round(4))

print("\nOverall:")
means = df.groupby('population')['delta_nll'].mean()
print(means.round(4))
print(f"\nDifference (intact - shuffled): {means.get('intact', 0) - means.get('shuffled', 0):.4f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'intact': '#2ecc71', 'shuffled': '#e74c3c'}

# Memory curves
ax = axes[0]
for pop in ['intact', 'shuffled']:
    pop_df = df[df['population'] == pop]
    means = pop_df.groupby('distance')['delta_nll'].mean()
    stds = pop_df.groupby('distance')['delta_nll'].std()
    sems = stds / np.sqrt(pop_df.groupby('distance').size())
    ax.errorbar(means.index, means.values, yerr=sems.values,
                marker='o', capsize=4, label=pop, color=colors[pop], linewidth=2)

ax.set_xlabel('Distance (tokens)')
ax.set_ylabel('Delta NLL')
ax.set_title('Memory Curves (Controlled Sampling)')
ax.legend()
ax.grid(True, alpha=0.3)

# Difference
ax = axes[1]
intact_means = df[df['population']=='intact'].groupby('distance')['delta_nll'].mean()
shuffled_means = df[df['population']=='shuffled'].groupby('distance')['delta_nll'].mean()
diff = intact_means - shuffled_means
colors_diff = ['#3498db' if d > 0 else '#e74c3c' for d in diff.values]
ax.bar(range(len(diff)), diff.values, color=colors_diff)
ax.set_xticks(range(len(diff)))
ax.set_xticklabels(diff.index)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Distance')
ax.set_ylabel('Intact - Shuffled')
ax.set_title('Difference (blue=intact higher, red=shuffled higher)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('lrtia_controlled.png', dpi=150)
plt.show()

In [ ]:
# Statistical test
from scipy import stats

print("\n" + "=" * 70)
print("STATISTICAL COMPARISON")
print("=" * 70)

for dist in sorted(df['distance'].unique()):
    intact_vals = df[(df['population']=='intact') & (df['distance']==dist)]['delta_nll']
    shuffled_vals = df[(df['population']=='shuffled') & (df['distance']==dist)]['delta_nll']
    
    if len(intact_vals) > 2 and len(shuffled_vals) > 2:
        t, p = stats.ttest_ind(intact_vals, shuffled_vals)
        diff = intact_vals.mean() - shuffled_vals.mean()
        sig = "*" if p < 0.05 else ""
        print(f"Distance {dist:3d}: intact={intact_vals.mean():.4f}, shuffled={shuffled_vals.mean():.4f}, diff={diff:+.4f}, p={p:.3f} {sig}")

In [ ]:
# Save
df.to_csv('lrtia_controlled_results.csv', index=False)
print("Saved to lrtia_controlled_results.csv")

try:
    from google.colab import files
    files.download('lrtia_controlled_results.csv')
    files.download('lrtia_controlled.png')
except:
    pass